In [1]:
import argparse
import requests
import os, glob
import xarray as xr
import numpy as np
from tqdm import tqdm

In [19]:
IC_time = '2024072100'

file = f'{IC_time[0:4]}/{IC_time[0:8]}/gfs.0p25.{IC_time}.f000.grib2'
IC_path = f'/wk2/yungyun/code_space/global_model_FCNV2_colab_education/input_data/ncep_data.grib2'


sfc_paramID = [165, 166, 167, 260074, 134, 3054, 228246, 228247]
sfc_datasets = None 
for sfc_i in sfc_paramID:
    temp = xr.open_dataset(
        IC_path,
        engine="cfgrib",
        filter_by_keys={'paramId': sfc_i},
    )
    sfc_datasets = xr.merge([sfc_datasets,temp],compat='override') if sfc_datasets != None else  temp
    # print(sfc_datasets.keys())
    
upper_paramID = [131, 132, 156, 157, 130]
upper_datasets = None 
for upper_i in upper_paramID:
    temp = xr.open_dataset(
        IC_path,
        engine="cfgrib",
        filter_by_keys={'typeOfLevel': 'isobaricInhPa', 'paramId': upper_i},
    )
    upper_datasets = xr.merge([upper_datasets,temp],compat='override') if upper_datasets != None else  temp
    upper_datasets =  upper_datasets.sortby('isobaricInhPa') 
upper_datasets['z'] = upper_datasets['gh']*9.8
upper_datasets = upper_datasets.drop_vars('gh')
upper_datasets = upper_datasets.rename({'isobaricInhPa':'level'})
upper_datasets =  upper_datasets.sortby('level') 
target_lev = [50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000]
upper_datasets = upper_datasets.sel(level=target_lev)


Ignoring index file '/wk2/yungyun/code_space/global_model_FCNV2_colab_education/input_data/ncep_data.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/wk2/yungyun/code_space/global_model_FCNV2_colab_education/input_data/ncep_data.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/wk2/yungyun/code_space/global_model_FCNV2_colab_education/input_data/ncep_data.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/wk2/yungyun/code_space/global_model_FCNV2_colab_education/input_data/ncep_data.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/wk2/yungyun/code_space/global_model_FCNV2_colab_education/input_data/ncep_data.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/wk2/yungyun/code_space/global_model_FCNV2_colab_education/input_data/ncep_data.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/wk2/yungyun/code_space/global_model_FCNV2_colab_education/input_data/ncep_data.grib2.5b7b6.idx' incomp

In [20]:
sfc_datasets

<xarray.Dataset> Size: 25MB
Dimensions:                (latitude: 721, longitude: 1440)
Coordinates:
  * latitude               (latitude) float64 6kB 90.0 89.75 ... -89.75 -90.0
  * longitude              (longitude) float64 12kB 0.0 0.25 0.5 ... 359.5 359.8
    time                   datetime64[ns] 8B ...
    step                   timedelta64[ns] 8B ...
    heightAboveGround      float64 8B ...
    valid_time             datetime64[ns] 8B ...
    meanSea                float64 8B ...
    surface                float64 8B ...
    atmosphereSingleLayer  float64 8B ...
Data variables:
    u10                    (latitude, longitude) float32 4MB -0.9895 ... -1.349
    v10                    (latitude, longitude) float32 4MB 0.9144 ... -5.466
    t2m                    (latitude, longitude) float32 4MB 273.2 ... 235.1
    prmsl                  (latitude, longitude) float32 4MB 1.008e+05 ... 1....
    sp                     (latitude, longitude) float32 4MB 1.008e+05 ... 6....
    pwat                   (latitude, longitude) float32 4MB 8.847 ... 0.3434
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2026-08-18T16:13 GRIB to CDM+CF via cfgrib-0.9.1...

In [ ]:
IC_path = f'/wk2/yungyun/code_space/global_model_FCNV2_colab_education/input_data/IFS_IC_raw_data.grib2'
# get data
sfc_paramID = [165, 166, 167, 168, 151, 134, 179, 137, 228, 228246, 228247]
sfc_datasets = None 
for sfc_i in sfc_paramID:
    temp = xr.open_dataset(IC_path, filter_by_keys={'paramId': sfc_i})
    sfc_datasets = xr.merge([sfc_datasets,temp],compat='override') if sfc_datasets != None else  temp
sfc_paramID = [131, 132]
for sfc_i in sfc_paramID:
    temp = xr.open_dataset(IC_path, filter_by_keys={'typeOfLevel': 'heightAboveGround','paramId': sfc_i})
    sfc_datasets = xr.merge([sfc_datasets,temp],compat='override') if sfc_datasets != None else  temp
sfc_datasets = sfc_datasets.assign_coords(longitude=(sfc_datasets.longitude % 360))
sfc_datasets = sfc_datasets.sortby('longitude')

upper_paramID = [157, 131, 132, 156, 135, 130]
upper_datasets = None 
for upper_i in upper_paramID:
    temp = xr.open_dataset(IC_path, filter_by_keys={'typeOfLevel': 'isobaricInhPa','paramId': upper_i})
    upper_datasets = xr.merge([upper_datasets,temp],compat='override') if upper_datasets != None else  temp
upper_datasets = upper_datasets.assign_coords(longitude=(upper_datasets.longitude % 360))
upper_datasets['z'] = upper_datasets['gh']*9.8
upper_datasets = upper_datasets.drop_vars('gh')
upper_datasets = upper_datasets.sortby('longitude')
upper_datasets = upper_datasets.rename({'isobaricInhPa':'level'})
upper_datasets =  upper_datasets.sortby('level') 


Ignoring index file '/wk2/yungyun/code_space/global_model_FCNV2_colab_education/input_data/IFS_IC_raw_data.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/wk2/yungyun/code_space/global_model_FCNV2_colab_education/input_data/IFS_IC_raw_data.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/wk2/yungyun/code_space/global_model_FCNV2_colab_education/input_data/IFS_IC_raw_data.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/wk2/yungyun/code_space/global_model_FCNV2_colab_education/input_data/IFS_IC_raw_data.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/wk2/yungyun/code_space/global_model_FCNV2_colab_education/input_data/IFS_IC_raw_data.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/wk2/yungyun/code_space/global_model_FCNV2_colab_education/input_data/IFS_IC_raw_data.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/wk2/yungyun/code_space/global_model_FCNV2_colab_education/input_da

In [24]:
sfc_paramID = [131, 132]
for sfc_i in sfc_paramID:
    temp = xr.open_dataset(IC_path, filter_by_keys={'typeOfLevel': 'heightAboveGround','paramId': sfc_i})
    sfc_datasets = xr.merge([sfc_datasets,temp],compat='override') if sfc_datasets != None else  temp
sfc_datasets = sfc_datasets.assign_coords(longitude=(sfc_datasets.longitude % 360))
sfc_datasets = sfc_datasets.sortby('longitude')
sfc_datasets

Ignoring index file '/wk2/yungyun/code_space/global_model_FCNV2_colab_education/input_data/IFS_IC_raw_data.grib2.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/wk2/yungyun/code_space/global_model_FCNV2_colab_education/input_data/IFS_IC_raw_data.grib2.5b7b6.idx' incompatible with GRIB file


<xarray.Dataset> Size: 37MB
Dimensions:            (latitude: 721, longitude: 1440)
Coordinates:
  * latitude           (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * longitude          (longitude) float64 12kB 0.0 0.25 0.5 ... 359.5 359.8
    time               datetime64[ns] 8B ...
    step               timedelta64[ns] 8B ...
    heightAboveGround  float64 8B ...
    valid_time         datetime64[ns] 8B ...
    meanSea            float64 8B ...
    surface            float64 8B ...
    nominalTop         float64 8B ...
    entireAtmosphere   float64 8B ...
Data variables:
    u10                (latitude, longitude) float32 4MB -0.7673 ... -6.627
    v10                (latitude, longitude) float32 4MB -1.396 -1.396 ... 1.448
    t2m                (latitude, longitude) float32 4MB 272.7 272.7 ... 228.8
    d2m                (latitude, longitude) float32 4MB 272.2 272.2 ... 225.1
    msl                (latitude, longitude) float32 4MB 1.008e+05 ... 1.006e+05
    sp                 (latitude, longitude) float32 4MB 1.008e+05 ... 6.857e+04
    ttr                (latitude, longitude) float32 4MB 0.0 0.0 0.0 ... 0.0 0.0
    tcwv               (latitude, longitude) float32 4MB 8.24 8.24 ... 0.3024
    tp                 (latitude, longitude) float32 4MB 0.0 0.0 0.0 ... 0.0 0.0
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-08-18T16:18 GRIB to CDM+CF via cfgrib-0.9.1...

In [27]:
temp = xr.open_dataset(IC_path, filter_by_keys={'typeOfLevel': 'heightAboveGround','paramId': 228246})
temp

Ignoring index file '/wk2/yungyun/code_space/global_model_FCNV2_colab_education/input_data/IFS_IC_raw_data.grib2.5b7b6.idx' incompatible with GRIB file


<xarray.Dataset> Size: 4MB
Dimensions:            (latitude: 721, longitude: 1440)
Coordinates:
  * latitude           (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * longitude          (longitude) float64 12kB -180.0 -179.8 ... 179.5 179.8
    time               datetime64[ns] 8B ...
    step               timedelta64[ns] 8B ...
    heightAboveGround  float64 8B ...
    valid_time         datetime64[ns] 8B ...
Data variables:
    u100               (latitude, longitude) float32 4MB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-08-18T16:20 GRIB to CDM+CF via cfgrib-0.9.1...

In [35]:
IC_time = '2026081600'
save_folder = f'input_data'
output_data = f'{save_folder}/IFS_initial_condition.npy'
os.makedirs(save_folder, exist_ok=True)

IC_path = f'{save_folder}/IFS_IC_raw_data.grib2'
if IC_time[-2:]=='06' or IC_time[-2:]=='18':
    download_name =f'https://storage.googleapis.com/ecmwf-open-data/{IC_time[:8]}/{IC_time[8:]}z/ifs/0p25/scda/{IC_time}0000-0h-scda-fc.grib2'
else:
    download_name =f'https://storage.googleapis.com/ecmwf-open-data/{IC_time[:8]}/{IC_time[8:]}z/ifs/0p25/oper/{IC_time}0000-0h-oper-fc.grib2'
    print(download_name)
response = requests.get(download_name)
with open(IC_path, "wb") as f:
    f.write(response.content)
    f.close()

# get data
sfc_paramID = [165, 166, 167, 168, 151, 134, 179, 137, 228, 228246, 228247]
sfc_datasets = None 
for sfc_i in sfc_paramID:
    temp = xr.open_dataset(IC_path, filter_by_keys={'paramId': sfc_i})
    sfc_datasets = xr.merge([sfc_datasets,temp],compat='override') if sfc_datasets != None else  temp
# sfc_paramID = [131, 132]
# for sfc_i in sfc_paramID:
#     temp = xr.open_dataset(IC_path, filter_by_keys={'typeOfLevel': 'heightAboveGround','paramId': sfc_i})
#     sfc_datasets = xr.merge([sfc_datasets,temp],compat='override') if sfc_datasets != None else  temp

sfc_datasets = sfc_datasets.assign_coords(longitude=(sfc_datasets.longitude % 360))
sfc_datasets = sfc_datasets.sortby('longitude')

upper_paramID = [157, 131, 132, 156, 135, 130]
upper_datasets = None 
for upper_i in upper_paramID:
    temp = xr.open_dataset(IC_path, filter_by_keys={'typeOfLevel': 'isobaricInhPa','paramId': upper_i})
    upper_datasets = xr.merge([upper_datasets,temp],compat='override') if upper_datasets != None else  temp
upper_datasets = upper_datasets.assign_coords(longitude=(upper_datasets.longitude % 360))
upper_datasets['z'] = upper_datasets['gh']*9.8
upper_datasets = upper_datasets.drop_vars('gh')
upper_datasets = upper_datasets.sortby('longitude')
upper_datasets = upper_datasets.rename({'isobaricInhPa':'level'})
upper_datasets =  upper_datasets.sortby('level') 

ordering = [ "10u",   "10v", "100u", "100v",   "2t",   "sp",  "msl", "tcwv",
            "u50",  "u100", "u150", "u200", "u250", "u300", "u400", "u500", "u600", "u700", "u850", "u925","u1000",
            "v50",  "v100", "v150", "v200", "v250", "v300", "v400", "v500", "v600", "v700", "v850", "v925","v1000",
            "z50",  "z100", "z150", "z200", "z250", "z300", "z400", "z500", "z600", "z700", "z850", "z925","z1000",
            "t50",  "t100", "t150", "t200", "t250", "t300", "t400", "t500", "t600", "t700", "t850", "t925","t1000",
            "r50",  "r100", "r150", "r200", "r250", "r300", "r400", "r500", "r600", "r700", "r850", "r925", "r1000"]

u10 = sfc_datasets.variables['u10']
v10 = sfc_datasets.variables['v10']
# u100 = sfc_datasets.variables['100u']
# v100 = sfc_datasets.variables['100v']
u100 = sfc_datasets.variables['u100']
v100 = sfc_datasets.variables['v100']
t2m = sfc_datasets.variables['t2m']
sp = sfc_datasets.variables['sp']
msl = sfc_datasets.variables['msl']
tcwv = sfc_datasets.variables['tcwv']
surface = np.stack([u10, v10, u100, v100, t2m, sp, msl, tcwv])

u = upper_datasets.variables['u']
v = upper_datasets.variables['v']
z = upper_datasets.variables['z']
t = upper_datasets.variables['t']
RH = upper_datasets.variables['r']
total_input = np.concatenate([surface, u, v, z, t, RH], axis=0)  



https://storage.googleapis.com/ecmwf-open-data/20260816/00z/ifs/0p25/oper/20260816000000-0h-oper-fc.grib2


Ignoring index file 'input_data/IFS_IC_raw_data.grib2.5b7b6.idx' older than GRIB file
Ignoring index file 'input_data/IFS_IC_raw_data.grib2.5b7b6.idx' older than GRIB file
Ignoring index file 'input_data/IFS_IC_raw_data.grib2.5b7b6.idx' older than GRIB file
Ignoring index file 'input_data/IFS_IC_raw_data.grib2.5b7b6.idx' older than GRIB file
Ignoring index file 'input_data/IFS_IC_raw_data.grib2.5b7b6.idx' older than GRIB file
Ignoring index file 'input_data/IFS_IC_raw_data.grib2.5b7b6.idx' older than GRIB file
Ignoring index file 'input_data/IFS_IC_raw_data.grib2.5b7b6.idx' older than GRIB file
Ignoring index file 'input_data/IFS_IC_raw_data.grib2.5b7b6.idx' older than GRIB file
Ignoring index file 'input_data/IFS_IC_raw_data.grib2.5b7b6.idx' older than GRIB file
Ignoring index file 'input_data/IFS_IC_raw_data.grib2.5b7b6.idx' older than GRIB file
Ignoring index file 'input_data/IFS_IC_raw_data.grib2.5b7b6.idx' older than GRIB file
Ignoring index file 'input_data/IFS_IC_raw_data.grib2.

In [40]:
upper_datasets

<xarray.Dataset> Size: 349MB
Dimensions:     (level: 14, latitude: 721, longitude: 1440)
Coordinates:
  * level       (level) float64 112B 10.0 50.0 100.0 150.0 ... 850.0 925.0 1e+03
  * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    time        datetime64[ns] 8B ...
    step        timedelta64[ns] 8B ...
    valid_time  datetime64[ns] 8B ...
Data variables:
    r           (level, latitude, longitude) float32 58MB 0.151 0.151 ... 99.92
    u           (level, latitude, longitude) float32 58MB -1.648 ... -2.413
    v           (level, latitude, longitude) float32 58MB 4.686 4.686 ... 4.811
    w           (level, latitude, longitude) float32 58MB -0.001292 ... -0.1044
    t           (level, latitude, longitude) float32 58MB 235.6 235.6 ... 248.5
    z           (level, latitude, longitude) float32 58MB 3.117e+05 ... -200.4
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-08-18T16:42 GRIB to CDM+CF via cfgrib-0.9.1...

In [41]:
sss = np.load('/wk2/yungyun/code_space/global_model_FCNV2_colab_education/input_data/IFS_initial_condition.npy')
sss.shape

(73, 721, 1440)